# 03 — EDA : justification des choix de modélisation

> Notebook **d'analyse exploratoire** dont chaque section répond à une question concrète qui conditionne notre modélisation :
> "Combien de NA ? Quelles distributions ? Quelle cardinalité ? Combien de lignes ?"
>
> Chaque section termine par un encadré **→ Implications modélisation** qui justifie un choix précis dans le notebook `04_Model_justification.ipynb`.

**Tâche primaire** : prédire **les nominés** (binaire : un film/personne est-il nominé dans cette catégorie cette année ?).
**Tâche bonus** : prédire **le gagnant** parmi les nominés.

⚠ **Note importante sur la donnée** : le dataset actuel `oscar_imdb_merged.parquet` ne contient **que des nominés** (2427 lignes). Pour la tâche "prédiction des nominés", il faudra construire un **set négatif** (films éligibles non nominés) — c'est l'objet de la §11.


**Périmètre final : 7 catégories** — Best Picture, Director, Actor, Actress, Supporting Actor, Supporting Actress, Original Screenplay. *Best Animated Feature* et *Best Visual Effects* ont été explorées puis **exclues** (faible valeur marché + pas de specs techniques).

## 1. Setup et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

DATA = Path('../Data/Processed/oscar_imdb_merged.parquet')
df = pd.read_parquet(DATA)

TARGETS = {
    'Best Actor in a Leading Role'       : 'Meilleur Acteur',
    'Best Actress in a Leading Role'     : 'Meilleure Actrice',
    'Best Actor in a Supporting Role'    : 'Meilleur Acteur SR',
    'Best Actress in a Supporting Role'  : 'Meilleure Actrice SR',
    'Best Picture'                       : 'Meilleur Film',
    'Best Directing'                     : 'Meilleur Réalisateur',
    'Best Animated Feature Film'         : 'Meilleur Film Animé',
    'Best Visual Effects'                : 'Meilleurs Effets Visuels',
    'Best Writing (Original Screenplay)' : 'Meilleur Scénario Original',
}

df_target = df[df['category'].isin(TARGETS)].copy()
print(f'Dataset complet  : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'Restreint aux 9 catégories cibles : {df_target.shape[0]} lignes')
print(f'Années couvertes : {df.year.min()} → {df.year.max()}')


: 

## 2. Vue d'ensemble du dataset

Avant tout, on regarde les types et un échantillon pour comprendre ce qu'on a en main.


In [ ]:
df.dtypes.to_frame('dtype').T

In [ ]:
df.head(3)

**→ Implications modélisation** :
- 30 colonnes, types **mixtes** : float, int, str, listes Python (`keywords`, `production_countries`).
- Les colonnes de listes ne peuvent pas être feedées directement → il faut les transformer (one-hot pour `production_countries`, TF-IDF/count pour `keywords`).
- `directors` et `writers` sont des chaînes pipe-separées (`nm123|nm456`) → il faudra `.str.split('|')` puis traiter chaque scénariste/réalisateur individuellement.


## 3. Valeurs manquantes — le tableau central

C'est la question N°1 : pour chaque feature, combien de NA ? Une feature avec > 50 % de NA est inutilisable, entre 5 et 50 % il faut imputer, en dessous de 5 % on peut ignorer ou imputer simplement.


In [ ]:
# Table NA globale (toutes catégories confondues)
na = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': df.isna().mean() * 100,
    'dtype': df.dtypes.astype(str),
})
na = na.sort_values('pct_missing', ascending=False)
na['pct_missing'] = na['pct_missing'].round(1)
na


In [ ]:
# Visualisation barplot des features avec NA > 0
na_plot = na[na.n_missing > 0].sort_values('pct_missing')
if len(na_plot):
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(na_plot))))
    ax.barh(na_plot.index, na_plot.pct_missing, color='salmon')
    ax.set_xlabel('% valeurs manquantes')
    ax.set_title('Features avec au moins une valeur manquante')
    for i, (idx, row) in enumerate(na_plot.iterrows()):
        ax.text(row.pct_missing + 0.5, i, f"{row.n_missing} ({row.pct_missing:.1f}%)",
                va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Aucune valeur manquante !')


**→ Implications modélisation** :

| Feature | % NA | Décision |
|---|---|---|
| `nconst` | ~73 % | **Attendu** : seulement renseigné pour catégories "personne" (acteurs, réalisateur, scénariste). À utiliser **conditionnellement** par catégorie. |
| `writers` | 2.3 % | Imputer "unknown" — négligeable. |
| `n_cast` | ~4 % | Imputer par la médiane (10). |
| `overview`, `tagline`, `keywords`, TMDb fields | < 1 % | Quelques films non trouvés sur TMDb. Filtrer ces lignes ou imputer chaîne vide. |
| `runtime_minutes`, `genres`, `imdb_rating` | < 0.5 % | Imputer médiane/mode, ou drop. |

→ **Aucune feature à supprimer** pour cause de NA excessifs. Le dataset est très propre. Cela permet d'éviter `IterativeImputer` complexe : simple `SimpleImputer(strategy='median')` ou chaîne vide suffit.


## 4. Couverture par catégorie cible

> 🔎 **Démarche : 9 catégories candidates explorées → 7 retenues.** On analyse ci-dessous l'ensemble des catégories envisagées. *Best Animated Feature* et *Best Visual Effects* sont **explorées ici mais exclues du livrable final** (faible valeur marché + aucune spec technique exploitable — équipes VFX, plans truqués — dans IMDb/TMDb). La modélisation porte donc sur les **7 catégories** restantes. *Garder leur exploration documente la décision, plutôt que de la masquer.*

Combien de lignes par catégorie ? Combien de winners ? Est-ce stable dans le temps ?


In [ ]:
# Vue par catégorie
rows = []
for cat_en, cat_fr in TARGETS.items():
    s = df[df.category == cat_en]
    rows.append({
        'Catégorie': cat_fr,
        'n': len(s),
        'winners': int(s.winner.sum()),
        'taux_winners': f'{s.winner.mean():.1%}',
        'années': f'{s.year.min()}-{s.year.max()}',
        'noms/an (mean)': round(s.groupby('year').size().mean(), 1),
    })
pd.DataFrame(rows)


In [ ]:
# Nominees par année pour chaque catégorie (drift du nombre de nominés)
fig, ax = plt.subplots(figsize=(11, 5))
for cat_en, cat_fr in TARGETS.items():
    s = df[df.category == cat_en].groupby('year').size()
    ax.plot(s.index, s.values, marker='o', markersize=3, label=cat_fr, alpha=0.8)
ax.set_xlabel('Année')
ax.set_ylabel('Nombre de nominés')
ax.set_title('Évolution du nombre de nominés par catégorie (2000-2026)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()


**→ Implications modélisation** :

- Les 7 catégories ont entre **101 et 201 lignes**. C'est **petit** → on est dans le régime "biais OK, variance critique" (session 02). Régularisation forte obligatoire, modèles complexes risquent l'overfit.
- **Best Picture est un cas à part** : passage de 5 à 10 nominés en 2009/2010 (réforme de l'Académie). Cela introduit une **dérive structurelle** dans le nombre de nominés → la base-rate "winner" baisse de 20 % à 10 %. À mentionner dans la soutenance, et à gérer via une feature `decade` ou un poids exponentiel sur les années récentes.
- Best Animated Feature démarre en 2002 (catégorie créée plus tard) → moins d'années, mais base-rate stable.
- → On garde **les 7 catégories séparément** plutôt qu'un modèle global avec `category` en feature : les dynamiques sont trop différentes (taille, drift, types de signaux).


## 5. Distributions des features numériques

Pour chaque feature numérique, on regarde si la distribution est gaussienne, log-normale, ou très étalée. Cela conditionne :
- **LR** : a besoin de features approximativement gaussiennes → log transform si étalé.
- **Arbres (RF/XGBoost/LightGBM)** : insensibles aux transformations monotones → n'ont pas besoin de log.


In [ ]:
num_cols = ['imdb_rating', 'imdb_votes', 'runtime_minutes',
            'budget', 'revenue', 'tmdb_vote_average', 'tmdb_vote_count']

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    ax = axes[i]
    serie = df[col].dropna()
    ax.hist(serie, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{col}\nmedian={serie.median():.1f}  max={serie.max():.0f}')
    ax.tick_params(labelsize=8)
axes[-1].axis('off')  # case vide
plt.tight_layout()
plt.show()


In [ ]:
# Skewness pour quantifier l'asymétrie
from scipy import stats
skew = pd.DataFrame({
    'feature': num_cols,
    'skewness': [df[c].dropna().skew() for c in num_cols],
    '%_zero': [(df[c] == 0).mean() * 100 for c in num_cols],
    'min_non_zero': [df.loc[df[c] > 0, c].min() if (df[c] > 0).any() else None for c in num_cols],
}).round(2)
skew['log_recommandé'] = skew['skewness'].abs() > 2
skew


In [ ]:
# Distributions après log transform
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for i, col in enumerate(['imdb_votes', 'budget', 'revenue', 'tmdb_vote_count']):
    ax = axes[i]
    serie = df[col].replace(0, np.nan).dropna()
    ax.hist(np.log1p(serie), bins=40, color='seagreen', edgecolor='white')
    ax.set_title(f'log1p({col})')
    ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()


**→ Implications modélisation** :

- `imdb_votes`, `budget`, `revenue`, `tmdb_vote_count` : **fortement asymétriques** (skewness > 5). Le log1p les rend quasi-gaussiennes.
- `budget` et `revenue` ont ~10-15 % de zéros qui sont en réalité des **inconnus** (TMDb sans donnée). Il faut les remplacer par NaN avant log, sinon `log1p(0) = 0` crée un cluster artificiel à 0.
- `imdb_rating`, `tmdb_vote_average`, `runtime_minutes` : distributions raisonnables, **pas besoin** de transformation.

→ **Recommandation concrète** pour la LR (modèle #1 du menu) : créer `log_budget`, `log_revenue`, `log_imdb_votes`, `log_tmdb_vote_count` en remplaçant les 0 par NaN d'abord. Pour les arbres : facultatif, n'apporte rien.


## 6. Cardinalité des features catégorielles

Pour chaque colonne catégorielle, combien de valeurs uniques ? Cela détermine la stratégie d'encodage :
- **Cardinalité faible (≤ 15)** : one-hot direct OK.
- **Cardinalité moyenne (15-50)** : one-hot ou target encoding (par fold).
- **Cardinalité haute (> 50)** : LightGBM `categorical_feature` natif, OU target encoding K-fold.


In [ ]:
# Cardinalité de toutes les colonnes catégorielles
def explode_pipe(serie):
    # Pour les colonnes pipe-separées comme `directors` ou `writers`.
    return serie.dropna().str.split('|').explode()

def explode_list(serie):
    # Pour les colonnes liste comme `keywords` ou `production_countries`.
    return serie.dropna().explode()

def explode_comma(serie):
    # Pour `genres` qui est string virgule-séparée.
    return serie.dropna().str.split(',').explode().str.strip()

cards = pd.DataFrame([
    ('genres (atomique)', explode_comma(df['genres']).nunique(), 'comma-sep'),
    ('directors (atomique)', explode_pipe(df['directors']).nunique(), 'pipe-sep'),
    ('writers (atomique)', explode_pipe(df['writers']).nunique(), 'pipe-sep'),
    ('original_language', df['original_language'].nunique(), 'str'),
    ('production_countries (atomique)', explode_list(df['production_countries']).nunique(), 'list[str]'),
    ('keywords (atomique)', explode_list(df['keywords']).nunique(), 'list[str]'),
    ('category', df['category'].nunique(), 'str'),
], columns=['feature', 'nunique', 'format'])
cards


In [ ]:
# Top occurrences pour les colonnes les plus stratégiques
print('--- TOP 10 GENRES ---')
print(explode_comma(df['genres']).value_counts().head(10).to_string())
print('\n--- TOP 10 LANGUES ---')
print(df['original_language'].value_counts().head(10).to_string())
print('\n--- TOP 10 PAYS DE PROD ---')
print(explode_list(df['production_countries']).value_counts().head(10).to_string())


In [ ]:
# Pour directors : distribution du nombre d'apparitions
dir_counts = explode_pipe(df['directors']).value_counts()
print(f'Nombre de réalisateurs uniques : {len(dir_counts)}')
print(f'Nb avec 1 seule apparition     : {(dir_counts == 1).sum()} ({(dir_counts == 1).mean():.0%})')
print(f'Nb avec ≥ 5 apparitions        : {(dir_counts >= 5).sum()}')
print(f'\nTop 10 réalisateurs (par nb de films présents dans le dataset) :')
print(dir_counts.head(10).to_string())


**→ Implications modélisation** :

| Feature | Cardinalité | Stratégie d'encodage |
|---|---|---|
| `genres` (atomique) | ~25 | **One-hot** des top 12, le reste en "other". Faible cardinalité, OK. |
| `original_language` | ~30 | **One-hot top 3 (en/fr/es) + `is_english`**. Distribution écrasée par "en" (~90 %). |
| `production_countries` | ~70 | **One-hot top 5 + `n_countries` + `is_coproduction`**. Le top 5 (US/UK/FR/DE/CA) couvre la majorité. |
| `directors` | **~700** | **LightGBM categorical natif** (modèle #4 du menu), OU **target encoding K-fold** pour les autres modèles. **One-hot impossible** : 700 features pour 128 lignes = catastrophe. |
| `writers` | **~1500** | Idem `directors`. La majorité apparaît une seule fois → peu d'informations exploitables, plutôt utiliser des **features dérivées** : `writer_prior_noms`. |
| `keywords` (atomique) | ~6000 | Pas une catégorielle à encoder — c'est du **texte**. TF-IDF + SVD (§7). |

→ **Le point critique** : la haute cardinalité de `directors` (~700) **justifie spécifiquement le choix de LightGBM** comme modèle pari pour la catégorie Best Directing. C'est l'avantage différenciant vs XGBoost (session 04).


## 7. Features texte : longueurs et richesse

Pour les catégories Best Picture, Scénario Original, Animated, on veut utiliser `keywords` et `overview`. Avant de monter une pipeline TF-IDF, on regarde la taille et la qualité de ces textes.


In [ ]:
# Longueur des textes
df_text = df_target.copy()
df_text['n_keywords'] = df_text['keywords'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
df_text['n_overview_words'] = df_text['overview'].fillna('').str.split().str.len()
df_text['n_tagline_words'] = df_text['tagline'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for i, col in enumerate(['n_keywords', 'n_overview_words', 'n_tagline_words']):
    ax = axes[i]
    ax.hist(df_text[col].dropna(), bins=30, color='mediumpurple', edgecolor='white')
    s = df_text[col].dropna()
    ax.set_title(f'{col}\nmédiane={s.median():.0f}  max={s.max():.0f}')
plt.tight_layout()
plt.show()

print('\nStatistiques résumées :')
df_text[['n_keywords', 'n_overview_words', 'n_tagline_words']].describe().round(1)


In [ ]:
# Top keywords globaux et chez les gagnants (toutes catégories)
from collections import Counter

all_kw = Counter()
win_kw = Counter()
for _, row in df_target.iterrows():
    kws = row['keywords'] if isinstance(row['keywords'], (list, np.ndarray)) else []
    all_kw.update(kws)
    if row['winner']:
        win_kw.update(kws)

# Lift : sur-représentation chez les gagnants
top_kw = pd.DataFrame({
    'keyword': list(all_kw.keys()),
    'count_total': list(all_kw.values()),
    'count_winners': [win_kw.get(k, 0) for k in all_kw],
})
top_kw['rate'] = top_kw['count_winners'] / top_kw['count_total']
top_kw['lift'] = top_kw['rate'] / df_target['winner'].mean()
top_kw = top_kw[top_kw['count_total'] >= 20]  # filtre keywords rares
top_kw = top_kw.sort_values('lift', ascending=False)

print('--- TOP 15 keywords sur-représentés chez les WINNERS (lift) ---')
print(top_kw.head(15).to_string(index=False))
print('\n--- TOP 15 keywords FRÉQUENTS globalement (volume) ---')
print(top_kw.sort_values('count_total', ascending=False).head(15)[['keyword', 'count_total', 'rate']].to_string(index=False))


**→ Implications modélisation** :

- **`keywords`** est riche : médiane ~16 mots-clés par film, max ~37. C'est la **principale source de texte exploitable**.
- **`overview`** est court (médiane ~30 mots, soit 1-3 phrases) → signal limité, à coupler avec `keywords` plutôt qu'utiliser seul.
- **`tagline`** est très court (~6 mots) et souvent marketing → peu de signal ML, ignorable.
- Certains keywords ont un **lift > 1.5** chez les gagnants (ex: "based on a novel", "biographical"), donc le signal existe — TF-IDF + SVD est justifié.
- → **Recommandation** : pour les catégories texte (Picture, Screenplay, Animated), combiner `keywords + overview` en un seul corpus, `TfidfVectorizer(max_features=300, min_df=2, ngram_range=(1,2))`, puis `TruncatedSVD(20)` avant d'envoyer à RF/XGBoost. Pour LR L2, on peut feeder le TF-IDF complet (la pénalité L2 fait la sélection).


## 8. Corrélations entre features numériques

Si deux features sont très corrélées (|r| > 0.85), on n'a pas besoin des deux pour la LR (multicolinéarité). Pour les arbres, c'est moins critique mais cela dilue les importances.


In [ ]:
num_cols2 = ['imdb_rating', 'tmdb_vote_average', 'imdb_votes', 'tmdb_vote_count',
             'runtime_minutes', 'budget', 'revenue', 'n_cast', 'n_genres']

# Log transform pour les colonnes très étalées
df_corr = df[num_cols2].copy()
for c in ['imdb_votes', 'tmdb_vote_count', 'budget', 'revenue']:
    df_corr[c] = np.log1p(df_corr[c].replace(0, np.nan))

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Corrélations entre features numériques (log appliqué quand utile)')
plt.tight_layout()
plt.show()


**→ Implications modélisation** :

- `imdb_rating` ↔ `tmdb_vote_average` : **forte corrélation** (~0.7-0.85). On peut garder les deux mais y ajouter `rating_gap = imdb_rating - tmdb_vote_average` qui isole l'écart critique/public.
- `imdb_votes` ↔ `tmdb_vote_count` : très corrélés (les deux mesurent la popularité). Garder un seul (préférer `log_imdb_votes`, plus stable historiquement).
- `budget` ↔ `revenue` : corrélés (~0.7) mais pas redondants — leur **ratio** (`roi`) est la feature dérivée utile.
- `n_cast`, `n_genres` : peu corrélés avec le reste → features autonomes utiles.

→ Pour la **LR** : ne pas mettre `imdb_votes` et `tmdb_vote_count` ensemble. Pour **RF/XGBoost/LightGBM** : pas critique, les arbres gèrent.


## 9. Dérive temporelle — features clés au fil du temps

La session 02 du cours insiste : si la distribution des features change entre 2000 et 2026, un modèle entraîné sur l'ensemble extrapolera mal vers 2027. On vérifie quelques features clés.


In [ ]:
key_feats = ['imdb_rating', 'runtime_minutes', 'budget', 'revenue']

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()
for i, col in enumerate(key_feats):
    ax = axes[i]
    sub = df[['year', col]].dropna()
    if col in ['budget', 'revenue']:
        sub = sub[sub[col] > 0]
        sub[col] = np.log1p(sub[col])
        title = f'log1p({col})'
    else:
        title = col
    yearly = sub.groupby('year')[col].agg(['mean', 'std']).reset_index()
    ax.plot(yearly['year'], yearly['mean'], marker='o', color='C0')
    ax.fill_between(yearly['year'],
                    yearly['mean'] - yearly['std'],
                    yearly['mean'] + yearly['std'],
                    alpha=0.2, color='C0')
    ax.set_title(f'Évolution de {title} (moyenne ± écart-type)')
    ax.set_xlabel('Année')
plt.tight_layout()
plt.show()


In [ ]:
# Mini PSI pour quantifier le drift sur deux périodes (2000-2015 vs 2016-2026)
def psi(expected, actual, bins=10):
    # Population Stability Index (cf. session 02).
    expected, actual = np.asarray(expected), np.asarray(actual)
    breakpoints = np.quantile(expected, np.linspace(0, 1, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    exp_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected) + 1e-6
    act_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual) + 1e-6
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

mid = 2015
psis = []
for col in ['imdb_rating', 'imdb_votes', 'runtime_minutes', 'budget', 'revenue']:
    before = df.loc[df.year <= mid, col].dropna()
    after  = df.loc[df.year > mid,  col].dropna()
    if col in ['imdb_votes', 'budget', 'revenue']:
        before, after = np.log1p(before.replace(0, np.nan).dropna()), np.log1p(after.replace(0, np.nan).dropna())
    val = psi(before, after)
    flag = 'OK' if val < 0.1 else 'moderate' if val < 0.25 else 'important'
    psis.append((col, round(val, 3), flag))
pd.DataFrame(psis, columns=['feature', 'PSI 2000-2015 vs 2016-2026', 'interprétation'])


**→ Implications modélisation** :

- **`imdb_votes` a un drift important** (PSI typiquement > 0.25) : c'est attendu, les films récents accumulent moins de votes que ceux qui ont 20 ans. Cela peut **biaiser** un modèle qui apprendrait "beaucoup de votes → nominé" sur 2000-2010 puis appliquerait ce critère à 2025.
- → **Solution** : remplacer `imdb_votes` brut par **`log_imdb_votes_z_by_decade`** (z-score par décennie) pour neutraliser l'effet âge.
- Les autres features sont relativement stables (`imdb_rating`, `runtime_minutes`).
- → **Décade comme feature** : justifié pour Best Picture (réforme 2009 du nb de nominés) et Animated (changement 2D→3D).


## 10. Déséquilibre de la cible (winner) par catégorie

Pour la tâche **bonus** (prédiction du gagnant parmi les nominés), on regarde le taux de winners par catégorie. Cela conditionne le besoin de `class_weight`, `scale_pos_weight`, ou de SMOTE.


In [ ]:
rows = []
for cat_en, cat_fr in TARGETS.items():
    s = df[df.category == cat_en]
    rows.append({'Catégorie': cat_fr, 'n': len(s), 'winners': int(s.winner.sum()),
                 'taux_winners': s.winner.mean(), 'taux_losers': 1 - s.winner.mean()})
imb = pd.DataFrame(rows).sort_values('taux_winners')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imb['Catégorie'], imb['taux_winners'], color='C0', label='winners')
ax.barh(imb['Catégorie'], imb['taux_losers'], left=imb['taux_winners'], color='lightgray', label='non-winners')
for i, row in enumerate(imb.itertuples()):
    ax.text(row.taux_winners / 2, i, f'{row.taux_winners:.0%}', va='center', ha='center', color='white', fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel('Proportion')
ax.set_title('Équilibre winner / non-winner par catégorie')
ax.legend(loc='center right')
plt.tight_layout()
plt.show()
imb.assign(taux_winners=lambda d: d['taux_winners'].map('{:.1%}'.format))


**→ Implications modélisation** :

- Le taux winners oscille entre **13.4 %** (Best Picture, le seul "vrai" déséquilibre) et **25.7 %** (VFX, quasi-équilibré).
- C'est un **déséquilibre modéré** — pas besoin de SMOTE (qui générerait du synthétique non représentatif sur 24-27 winners).
- → **Stratégie simple** :
  - LR, RF, LightGBM : `class_weight='balanced'`
  - XGBoost : `scale_pos_weight = (#non-winners) / (#winners)` (typiquement 3-7)
- → Best Picture étant le plus déséquilibré, **c'est là qu'on doit le plus surveiller les métriques par classe** (PR-AUC plus que ROC-AUC).


## 11. ⚠ Le problème central : construire le set négatif pour la prédiction des nominés

C'est ici que la tâche **primaire** (prédire les nominés) diverge complètement de la tâche **bonus** (prédire les gagnants).

**Le constat** : notre dataset contient `2427 lignes — toutes positives` (= toutes des nominations). Pour faire de la classification binaire "nominé / non nominé", il **manque les négatifs** : les films/personnes qui étaient éligibles cette année mais n'ont pas été nominés.

### Les stratégies possibles pour construire le set négatif (A · B · B+ · C)

| Stratégie | Description | Avantage | Inconvénient |
|---|---|---|---|
| **A. Tous films IMDb par année** | Tout film `titleType=movie`, `startYear=year-1` | Exhaustif | ~3 000-5 000 films/an → déséquilibre ~0.1 % impraticable |
| **B. Films pré-filtrés "in conversation"** | Films IMDb `numVotes ≥ 5000`, `runtime ≥ 60`, movie (proxy d'éligibilité Oscar) | Pool raisonnable (~150-300/an), déséquilibre ~2-5 % | Filtre arbitraire à justifier |
| **B+. B durcie — _implémentée_** | B + `numVotes ≥ 10 000`, `averageRating ≥ 5.0`, hors-documentaire ; univers **étendu aux personnes** (acteurs/réalisateurs des films éligibles via `principals`) ; **ré-injection de 100 % des vrais nominés** | Base-rate réaliste ~1-3 %, **aucun positif perdu**, logique experte + data-driven | Seuils à assumer (justifiés §2 du notebook 05) |
| **C. Films ayant gagné ≥ 1 award précurseur** | Golden Globes, BAFTA, SAG, Critics Choice (à scraper) | Pool très pertinent | Données non disponibles dans notre repo |

**Recommandation** : **Stratégie B**, durcie en **B+** (ligne ci-dessus), implémentée dans `05_Nomination_modeling.ipynb`. Filtre de base (B) :

```python
elig = imdb_basics[
    (imdb_basics.titleType == 'movie') &
    (imdb_basics.startYear.between(1999, 2025)) &
    (imdb_basics.runtimeMinutes >= 60) &
    (imdb_basics.tconst.isin(imdb_ratings.query('numVotes >= 5000').tconst))
]
```

### Impact théorique sur les modèles

| Aspect | Tâche WINNER (bonus) | Tâche NOMINEE (primaire) |
|---|---|---|
| Cible | winner=1 parmi nominés | nominated=1 parmi éligibles |
| n par catégorie | 100-200 (existant) | ~3000-6000 après ajout des négatifs |
| Base rate | 13-26 % | ~2-8 % |
| Évaluation | top-1 par (cat, year) | top-K par (cat, year), avec K = nb de slots |
| Modèle théorique gagnant | LR L2 souvent (n petit, signal linéaire) | **XGBoost/LightGBM** plus systématiquement (n plus grand, signal plus diffus) |
| Métrique principale | top-1 accuracy par groupe | **Precision@K** + Recall@K par groupe |
| Calibration | utile pour Streamlit | **critique** : on affiche "P(nomination) = X %" |

### → Implications pour le notebook `04_Model_justification.ipynb`

Le menu des 5 modèles **reste pertinent** pour les deux tâches, mais le **classement attendu se déplace vers les boosting** quand on passe à la prédiction des nominés :
- Tâche **nominee** : XGBoost et LightGBM dominent **plus souvent** (n plus grand, déséquilibre plus fort, plus de features hétérogènes à digérer).
- Tâche **winner** : LR L2 reste compétitive grâce à la petite taille (cf. matrice §14 du notebook de justification).

→ Concrètement, ne pas reconstruire un nouveau menu : garder les 5 mêmes modèles, **lancer la boucle d'évaluation deux fois** (cible=nominated, cible=winner conditionnellement aux nominés), comparer les classements.


> ✅ **Réalisé** — le set négatif (stratégie B+ raffinée : votes≥10000, rating≥5.0, hors-doc) est construit dans `05_Nomination_modeling.ipynb` sur les **7 catégories** (film + personne via `principals`). Base-rate réel ~1–3 %, Precision@K **~0,33–0,69** (meilleur modèle **variable par catégorie** — LogReg, RandomForest, AdaBoost, XGBoost — parmi **7 modèles** benchmarkés en GroupKFold), lift **×14 à ×78** vs hasard.

## 12. Synthèse — ce que cette EDA nous dit pour la modélisation

| Constat (chiffré) | Décision modélisation |
|---|---|
| Aucune feature avec > 5 % de NA hors `nconst` (catégorie-dépendant) | Imputation simple suffit (median / "unknown") — pas d'IterativeImputer. |
| n = 101-201 lignes par catégorie | Régularisation forte ; **GroupKFold(5, groups=year)** ; aucune optimisation de grille > 12 combinaisons. |
| `imdb_votes`, `budget`, `revenue` skewness > 5 | **log1p obligatoire** pour la LR ; facultatif pour les arbres. |
| `budget`/`revenue` ont 10-15 % de zéros = inconnus | Remplacer 0 par NaN **avant** log. |
| `directors` cardinalité ~700, `writers` ~1500 | **LightGBM avec `categorical_feature`** pour Best Directing ; target encoding K-fold sinon ; **one-hot interdit**. |
| `genres` cardinalité ~25, `original_language` dominé par "en" | One-hot top-12 genres, `is_english` binaire. |
| `keywords` médian = 16 par film, `overview` ~30 mots | TF-IDF(`keywords` + `overview`) → TruncatedSVD(20) pour RF/XGBoost ; TF-IDF direct pour LR. |
| `imdb_rating` ↔ `tmdb_vote_average` corrélés (~0.85) | Garder un seul + `rating_gap` ; sinon multicolinéarité LR. |
| `imdb_votes` PSI > 0.25 entre 2000-2015 et 2016-2026 | Z-score `log_imdb_votes` **par décennie** ; ajouter `decade` en feature catégorielle. |
| Best Picture passe de 5 à 10 nominés en 2009/2010 | Feature `decade` ou `is_post_reform`. |
| Winner imbalance modéré (13-26 %) | `class_weight='balanced'` / `scale_pos_weight` ; **pas de SMOTE**. |
| Dataset = nominés uniquement | **Tâche nominee = à construire** : enrichir avec set négatif (stratégie B §11). |

### Implications opérationnelles pour la modélisation
1. **Ajouter dans `05_Nomination_modeling.ipynb`** : construction du set négatif (films IMDb éligibles non nominés, filtrés par `numVotes ≥ 5000`).
2. **Ajouter la feature `film_n_total_noms`** (nb total de nominations d'un film cette année) — utile partout sauf Best Picture lui-même.
3. **Ajouter la colonne `production_companies`** via TMDb — critique pour Best Animated Feature.
4. **Construire les features historiques personne** avec `groupby(nconst).cumcount().shift(1)` pour éviter le leakage temporel (cf. notebook de justification §1).
5. **Lancer la boucle d'évaluation `04_Model_justification.ipynb` §16** sur les 7 catégories × 5 modèles × 2 cibles (nominee + winner).
